# Training a Genova Model

This notebook walks through:

1. Configuring the model and training
2. Preprocessing synthetic data
3. Training a small model
4. Evaluating the trained model

We use synthetic data throughout -- no genome files required.

## 1. Configuration

In [ ]:
import torch
import numpy as np

from genova.utils.config import GenovaConfig
from genova.utils.reproducibility import set_seed

set_seed(42)

config = GenovaConfig()

# Small model for demo
config.model.d_model = 128
config.model.n_heads = 4
config.model.n_layers = 2
config.model.d_ff = 512
config.model.dropout = 0.1

# Training params
config.training.lr = 1e-3
config.training.epochs = 5
config.training.warmup_steps = 10
config.data.batch_size = 8
config.data.seq_length = 128

print("Model config:")
print(f"  d_model={config.model.d_model}, n_layers={config.model.n_layers}")
print(f"  n_heads={config.model.n_heads}, d_ff={config.model.d_ff}")
print(f"Training config:")
print(f"  lr={config.training.lr}, epochs={config.training.epochs}")

## 2. Create Synthetic Dataset

In [ ]:
from genova.data.tokenizer import GenomicTokenizer

tokenizer = GenomicTokenizer(mode="kmer", k=6, stride=1)
tokenizer.build_vocab()
config.model.vocab_size = tokenizer.vocab_size

# Generate synthetic sequences
def generate_random_dna(length: int, gc_bias: float = 0.5) -> str:
    bases = []
    for _ in range(length):
        if np.random.random() < gc_bias:
            bases.append(np.random.choice(['G', 'C']))
        else:
            bases.append(np.random.choice(['A', 'T']))
    return ''.join(bases)

num_train = 200
num_val = 50
seq_len = 128

train_seqs = [generate_random_dna(seq_len) for _ in range(num_train)]
val_seqs = [generate_random_dna(seq_len) for _ in range(num_val)]

print(f"Train sequences: {num_train}")
print(f"Val sequences: {num_val}")
print(f"Example: {train_seqs[0][:50]}...")

## 3. Prepare DataLoaders

In [ ]:
from torch.utils.data import Dataset, DataLoader

class SimpleGenomicDataset(Dataset):
    def __init__(self, sequences, tokenizer, max_length=128, mask_prob=0.15):
        self.sequences = sequences
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.mask_prob = mask_prob

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        tokens = self.tokenizer.encode(seq)
        tokens = tokens[:self.max_length]
        # Pad
        pad_len = self.max_length - len(tokens)
        attention_mask = [1] * len(tokens) + [0] * pad_len
        tokens = tokens + [0] * pad_len

        # MLM: mask some tokens
        labels = list(tokens)
        input_ids = list(tokens)
        for i in range(len(input_ids)):
            if attention_mask[i] == 1 and np.random.random() < self.mask_prob:
                input_ids[i] = 4  # mask token
            else:
                labels[i] = -100  # ignore in loss

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

train_ds = SimpleGenomicDataset(train_seqs, tokenizer)
val_ds = SimpleGenomicDataset(val_seqs, tokenizer)

train_loader = DataLoader(train_ds, batch_size=config.data.batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=config.data.batch_size)

batch = next(iter(train_loader))
print(f"Batch input_ids shape: {batch['input_ids'].shape}")
print(f"Batch labels shape: {batch['labels'].shape}")

**Expected output:**
```
Batch input_ids shape: torch.Size([8, 128])
Batch labels shape: torch.Size([8, 128])
```

## 4. Train the Model

In [ ]:
from genova.models.model_factory import create_model, count_parameters

model = create_model(config.model, task="mlm")
print(f"Parameters: {count_parameters(model):,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=config.training.lr, weight_decay=0.01)
loss_fn = torch.nn.CrossEntropyLoss(ignore_index=-100)

# Training loop
model.train()
for epoch in range(config.training.epochs):
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        )
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs
        loss = loss_fn(logits.view(-1, logits.size(-1)), batch["labels"].view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{config.training.epochs} - Loss: {avg_loss:.4f}")

**Expected output:**
```
Epoch 1/5 - Loss: ~8.3
Epoch 2/5 - Loss: ~7.9
Epoch 3/5 - Loss: ~7.5
Epoch 4/5 - Loss: ~7.2
Epoch 5/5 - Loss: ~6.8
```
(Loss should decrease over epochs.)

## 5. Evaluate

In [ ]:
model.eval()
val_loss = 0
correct = 0
total = 0

with torch.no_grad():
    for batch in val_loader:
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        )
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs
        loss = loss_fn(logits.view(-1, logits.size(-1)), batch["labels"].view(-1))
        val_loss += loss.item()

        # Accuracy on masked tokens
        mask = batch["labels"] != -100
        if mask.any():
            preds = logits.argmax(dim=-1)
            correct += (preds[mask] == batch["labels"][mask]).sum().item()
            total += mask.sum().item()

avg_val_loss = val_loss / len(val_loader)
accuracy = correct / total if total > 0 else 0

print(f"Validation Loss: {avg_val_loss:.4f}")
print(f"MLM Accuracy: {accuracy:.4f}")
print(f"Perplexity: {np.exp(avg_val_loss):.2f}")

**Expected output:**
```
Validation Loss: ~7.0
MLM Accuracy: ~0.01 (random init on synthetic data)
Perplexity: ~1100
```

(With a tiny model on random data, accuracy will be low. Real training on genomic data achieves much better results.)

## Summary

You learned how to:
- Configure model architecture and training hyperparameters
- Create synthetic genomic datasets with MLM masking
- Train a Genova model with gradient clipping
- Evaluate loss, accuracy, and perplexity

Next: See `03_variant_analysis.ipynb` for variant effect analysis.